# 01 — Data Extraction

Download the raw StatsBomb event and lineup parquets that feed the rest of the pipeline.

This notebook is a thin wrapper around `src.extraction.extract_and_save`. It is **idempotent**: files already present in `data/raw/` are skipped, so re-running the notebook after a partial failure resumes where it stopped.

**Expected runtime on a fresh clone:** 30–60 minutes for the five 2015/16 leagues, dominated by the HTTP fetches against the StatsBomb open-data CDN. Subsequent runs complete in seconds (only the skip checks run).

**Output:** ten parquet files in `data/raw/` — five `events_*` and five `lineups_*`, one of each per competition / season.

## Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.extraction import (
    extract_and_save,
    load_raw_events,
    load_raw_lineups,
    COMPETITIONS_TO_USE,
    DATA_RAW,
)

print(f'Project root      : {PROJECT_ROOT}')
print(f'Raw data target   : {DATA_RAW}')
print()
print('Competitions to extract:')
for c in COMPETITIONS_TO_USE:
    print(f"  - {c['name']:<32s}  competition_id={c['competition_id']:<3d}  season_id={c['season_id']}")

## Run the extraction

Calling `extract_and_save()` with no argument uses the default competition list defined in `src/extraction.py`. To customise, pass a list of dicts with the keys `name`, `competition_id`, `season_id`. Use `00_check_data.ipynb` to discover the StatsBomb identifiers.

In [ ]:
extract_and_save()

## Verify what was written

Quick sanity check on the row counts and the event-type distribution. A 2015/16 Big-Five league season typically produces 1.0–1.4 M events; the five leagues combined should land near 6 M events.

In [ ]:
events = load_raw_events()
lineups = load_raw_lineups()

print(f'Events  : {len(events):>9,d} rows')
print(f'Lineups : {len(lineups):>9,d} rows')
print()
print('Top event types:')
print(events['type'].value_counts().head(10).to_string())

## Next step

Run `notebooks/02_features.ipynb` to explore the per-player feature table, or call `src.features.build_features()` directly to regenerate `data/processed/player_features.parquet`.